In [13]:
import dagshub
dagshub.init(repo_owner='Sovith07', repo_name='yt_comment_analyzer', mlflow=True)

Initialized MLflow to track repo "Sovith07/yt_comment_analyzer"

Repository Sovith07/yt_comment_analyzer initialized!

In [14]:
import mlflow

mlflow.set_tracking_uri("https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow")

In [15]:
# Set or create an experiment
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

<Experiment: artifact_location='mlflow-artifacts:/d16facc30aa04e50b0ad2d54d92b94ef', creation_time=1786900713337, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1786900713337, lifecycle_stage='active', name='Exp 4 - Handling Imbalanced Data', tags={}, trace_location=None, workspace='default'>

In [16]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [17]:
df = pd.read_csv(r'D:\vs code projects\yt_comment_analyzer\data\processed\final_data.csv')
df.shape

(199508, 2)

In [ ]:
# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 3)  # Trigram setting
    max_features = 10000  # Set max_features to 10000 for TF-IDF

    # Step 4: Train-test split before vectorization and resampling
    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    # Step 2: Vectorization using TF-IDF, fit on training data only
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
    X_test_vec = vectorizer.transform(X_test)  # Transform test data

    # Step 3: Handle class imbalance based on the selected method (only applied to the training set)
    if imbalance_method == 'class_weights':
        # Use class_weight in Random Forest
        class_weight = 'balanced'
    else:
        class_weight = None  # Do not apply class_weight if using resampling

        # Resampling Techniques (only apply to the training set)
        if imbalance_method == 'oversampling':
            smote = SMOTE(random_state=42)
            X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'adasyn':
            adasyn = ADASYN(random_state=42)
            X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'smote_enn':
            smote_enn = SMOTEENN(random_state=42)
            X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    # Step 5: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight)
        model.fit(X_train_vec, y_train)

        # Step 6: Make predictions and log metrics
        y_pred = model.predict(X_test_vec)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig("D:\\vs code projects\\yt_comment_analyzer\\reports\\confusion_matrix_filename")
        mlflow.log_artifact("D:\\vs code projects\\yt_comment_analyzer\\artifacts\\confusion_matrix_filename")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}")

# Step 7: Run experiments for different imbalance methods
imbalance_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']

for method in imbalance_methods:
    run_imbalanced_experiment(method)

2026/08/18 12:32:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/18 12:32:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3/runs/d7cb5a8d76d74bf5a86e57034c750b05
🧪 View experiment at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3


2026/08/18 12:36:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/18 12:36:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3/runs/803a8bfbdf5a44c2aadfecca84447c94
🧪 View experiment at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3


2026/08/18 12:42:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/18 12:43:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3/runs/24c7cdd565d4489d9b1f242eb1855742
🧪 View experiment at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3


2026/08/18 12:45:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/18 12:45:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3/runs/7dd49a8df6774ebdbbc992c9152a50ac
🧪 View experiment at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3


2026/08/18 13:00:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/18 13:00:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_smote_enn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3/runs/5d209db3b2304dc08746fe1fb3db7875
🧪 View experiment at: https://dagshub.com/Sovith07/yt_comment_analyzer.mlflow/#/experiments/3
